# Evaluation - Robustness to perturbations

Accuracy vs increasing perturbation strength (JPEG compression, Gaussian blur, downsampling, additive noise) on a test subsample. Covers the image-input pipelines.

**Sections:** 0 Setup - 1 Sweep - 2 Curves - 3 Robustness summary

> Reads each pipeline's committed artifacts and reconstructs trained models via `utils.eval_protocols` (rebuilt from `best_params.json`). Run after training completes.

## 0 - Setup

In [ ]:
import sys, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

_here = Path.cwd()
_nb = _here if (_here / "utils").is_dir() else _here / "notebooks"
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))

from utils import eval_protocols as EP, metrics as Me, viz as V, datasets as D, explain as E, eda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = EP.ART / "evaluation" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
TAB = EP.ART / "evaluation"
PIPES = EP.available()
print("device:", device)
print("pipelines with trained results:", PIPES)
IMG_PIPES = [n for n in PIPES if EP.SPEC[n]["family"] == "image"]
SUBSAMPLE = 1500   # test images per perturbation level (speed)
print("image-input pipelines:", IMG_PIPES)
print("perturbations:", {k: v for k, v in EP.PERTURBATIONS.items()})

> **Scope:** the sweep covers the **image-input** pipelines. `clip-probe`, `patch-ensemble` and `dire-recon` use specialized inputs (cached embeddings / patch bags / diffusion-reconstruction maps), so they're omitted from the pixel-perturbation sweep here.

## 1 - Run the perturbation sweep

For each pipeline, each perturbation type, each strength level: build a perturbed loader and measure accuracy. (GPU; ~minutes per pipeline.)

In [ ]:
results = {kind: {n: [] for n in IMG_PIPES} for kind in EP.PERTURBATIONS}
for n in IMG_PIPES:
    model = EP.load_model(n, device)
    for kind, levels in EP.PERTURBATIONS.items():
        accs = []
        for lv in levels:
            r = EP.robustness_point(n, model, kind, lv, device, subsample=SUBSAMPLE)
            accs.append(r["accuracy"])
        results[kind][n] = accs
        print(f"{n:16s} {kind:20s} {[round(a,3) for a in accs]}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2 - Robustness curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9)); axes = axes.reshape(-1)
for ax, (kind, levels) in zip(axes, EP.PERTURBATIONS.items()):
    for n in IMG_PIPES:
        ax.plot(levels, results[kind][n], marker="o", ms=4, label=n)
    ax.set_title(kind); ax.set_xlabel("strength (" + kind + ")"); ax.set_ylabel("accuracy"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.suptitle("Robustness: accuracy vs perturbation strength", fontsize=13); fig.tight_layout()
fig.savefig(OUT / "robustness_curves.png", dpi=150, bbox_inches="tight"); plt.show()

## 3 - Robustness summary (mean accuracy across all perturbed conditions)

In [ ]:
rows = []
for n in IMG_PIPES:
    allv = [a for kind in EP.PERTURBATIONS for a in results[kind][n]]
    clean = results["jpeg_quality"][n][0]  # quality 100 ~ clean
    rows.append({"pipeline": n, "clean_acc": clean, "mean_perturbed_acc": float(np.mean(allv)), "worst_acc": float(np.min(allv))})
rob = pd.DataFrame(rows).sort_values("mean_perturbed_acc", ascending=False).reset_index(drop=True)
display(rob.round(4)); rob.round(6).to_csv(TAB / "robustness_summary.csv", index=False)
print("Most robust:", rob.iloc[0]["pipeline"], "| least:", rob.iloc[-1]["pipeline"])